# 🎭 脫口秀幽默預測系統 — Colab 訓練

## 設計原則
- ✅ **一部影片處理完立即刪除**，避免爆空間
- ✅ **定期存到 Google Drive**，防止 Colab 超時斷線
- ✅ **自動 Heartbeat 防止 idle timeout**
- ✅ **斷點續傳**：重新執行會跳過已處理的影片

## 流程
```
YouTube URL 清單
  ↓ 下載音訊 (yt-dlp)
  ↓ 取字幕 / Whisper ASR
  ↓ 笑聲偵測 (YAMNet)
  ↓ Setup-Punchline 對齊
  ↓ 自動標註 (Humor Score)
  ↓ 儲存到 Google Drive ← 每部影片後
  ↓ 刪除原始音訊 ← 節省空間
  ↓ 訓練 Reward Model (批次模式)
```

## Step 0: 掛載 Google Drive & 防 Timeout

In [ ]:
# ── 0-A. 掛載 Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/humor_bot'  # 改成你的 Drive 路徑
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'✅ Drive 掛載完成，資料存放於: {DRIVE_ROOT}')

In [ ]:
# ── 0-B. 防 Idle Timeout (Heartbeat + JavaScript 點擊) ────────
# Colab 在 90 分鐘無互動後會斷線。以下兩招並用：
# 1. JavaScript 自動點擊 (讓它以為有人在用)
# 2. Python keepalive 執行緒 (背景每 60 秒 ping)

from IPython.display import display, Javascript
import threading
import time
import subprocess

# JavaScript 解法：自動點擊「連線」按鈕，讓 Colab 覺得你還活著
display(Javascript('''
function keepAlive() {
    // 找「連線」按鈕 (不同版本 Colab 按鈕 selector 不同，試多種)
    const btns = [
        document.querySelector('colab-connect-button'),
        document.querySelector('paper-icon-button#toggleConnectButton'),
    ];
    // 只是製造滑鼠移動事件，不真正點擊
    document.dispatchEvent(new MouseEvent('mousemove'));
    console.log('[KeepAlive] ping at ' + new Date().toLocaleTimeString());
}
// 每 50 秒執行一次（比 Colab idle timeout 60s 更短）
window._keepAliveInterval = setInterval(keepAlive, 50000);
console.log('[KeepAlive] 已啟動！每 50 秒自動 ping');
'''))

# Python 層面：背景執行緒每 60 秒呼叫一個無害的系統指令
_stop_heartbeat = threading.Event()

def _heartbeat_worker():
    while not _stop_heartbeat.is_set():
        try:
            # 呼叫 nvidia-smi 確保 GPU context 活著（同時打印 GPU 使用率）
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
                 '--format=csv,noheader'],
                capture_output=True, text=True, timeout=10
            )
            gpu_info = result.stdout.strip() if result.returncode == 0 else 'N/A'
            print(f'[Heartbeat {time.strftime("%H:%M:%S")}] GPU: {gpu_info}')
        except Exception:
            pass
        _stop_heartbeat.wait(60)  # 等待 60 秒或直到 stop 信號

_heartbeat_thread = threading.Thread(target=_heartbeat_worker, daemon=True)
_heartbeat_thread.start()
print('✅ Heartbeat 已啟動（每 60 秒顯示 GPU 狀態）')
print('   若要停止，執行: _stop_heartbeat.set()')

## Step 1: 安裝套件

In [ ]:
# ── 1. 安裝所有依賴 ──────────────────────────────────────────
# 注意：Colab 重啟後需要重新安裝

print('📦 安裝核心套件...')
!pip install -q yt-dlp faster-whisper soundfile librosa
!pip install -q tensorflow tensorflow-hub  # YAMNet 笑聲偵測
!pip install -q sentence-transformers transformers datasets accelerate peft trl
!pip install -q opencv-python-headless     # 無 GUI 版本（Colab 適用）
!pip install -q chromadb                   # RAG 知識庫
!pip install -q openai                     # GPT-4o 分析
!pip install -q numpy pandas scipy matplotlib tqdm rich

# ffmpeg (下載音訊需要)
!apt-get install -qq -y ffmpeg > /dev/null 2>&1

print('✅ 套件安裝完成！')

## Step 2: Clone 專案並設定路徑

In [ ]:
# ── 2. 掛載專案 ──────────────────────────────────────────────
# 方法 A：直接從 Google Drive 複製專案（推薦，不需要 Git）
# 假設你已將整個 humor/ 資料夾同步到 Drive

import subprocess, sys, os

# 從 Drive 複製最新的 src/ 到 Colab 本地
DRIVE_SRC = f'{DRIVE_ROOT}/src'
LOCAL_PROJECT = '/content/humor'

if os.path.exists(f'{DRIVE_ROOT}/src'):
    # 從 Drive 同步
    !cp -r {DRIVE_ROOT}/src /content/humor_src
    sys.path.insert(0, '/content/humor_src')
    print('✅ 已從 Drive 載入專案 src')
else:
    # 方法 B：直接在 Colab 貼上 src/ 目錄（若無 Git）
    print('⚠️  請將 humor/src 資料夾上傳或同步到 Google Drive 的 humor_bot/ 目錄下')
    print('   或使用 Git clone:')
    print('   !git clone https://github.com/YOUR_REPO/humor.git /content/humor')
    print('   sys.path.insert(0, "/content/humor/src")')

# 設定工作目錄
DATA_DIR = '/content/data'  # Colab 本地暫存（重啟後消失，但節省 Drive 空間）
PROCESSED_DIR = f'{DRIVE_ROOT}/processed'  # 處理結果存 Drive（永久保存）
MODEL_DIR = f'{DRIVE_ROOT}/models'         # 模型存 Drive

for d in [DATA_DIR, PROCESSED_DIR, MODEL_DIR,
          f'{DATA_DIR}/raw/audio', f'{DATA_DIR}/raw/transcripts']:
    os.makedirs(d, exist_ok=True)

print(f'✅ 資料夾結構:')
print(f'   暫存音訊: {DATA_DIR}/raw/audio/ (每部影片完成後自動刪除)')
print(f'   永久結果: {PROCESSED_DIR}/')
print(f'   模型存放: {MODEL_DIR}/')

## Step 3: 核心工具函式

In [ ]:
# ── 3. 核心管線函式 ──────────────────────────────────────────
import json
import logging
import shutil
import time
import traceback
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger('humor_colab')


def load_processed_ids(log_path: str) -> set:
    """載入已處理的影片 ID（斷點續傳）"""
    if os.path.exists(log_path):
        with open(log_path, 'r') as f:
            return set(json.load(f))
    return set()


def save_processed_id(video_id: str, log_path: str):
    """記錄已處理的影片 ID"""
    ids = load_processed_ids(log_path)
    ids.add(video_id)
    with open(log_path, 'w') as f:
        json.dump(list(ids), f)


def cleanup_video_files(audio_path: str | None, subtitle_path: str | None):
    """處理完畢後刪除原始音訊與字幕，節省 Colab 空間"""
    deleted = []
    for p in [audio_path, subtitle_path]:
        if p and os.path.exists(p):
            os.unlink(p)
            deleted.append(os.path.basename(p))
    # 也嘗試刪除同目錄下的其他字幕變體
    if audio_path:
        audio_dir = Path(audio_path).parent
        video_id = Path(audio_path).stem
        for f in audio_dir.glob(f'{video_id}.*'):
            if f.is_file():
                f.unlink()
                deleted.append(f.name)
    if deleted:
        logger.info(f'🧹 已清理: {deleted}')


def sync_to_drive(local_path: str, drive_path: str):
    """將本地結果同步到 Google Drive"""
    try:
        os.makedirs(os.path.dirname(drive_path), exist_ok=True)
        shutil.copy2(local_path, drive_path)
        logger.info(f'☁️  已同步到 Drive: {os.path.basename(drive_path)}')
    except Exception as e:
        logger.warning(f'Drive 同步失敗: {e}')


def get_disk_usage_gb(path: str = '/content') -> float:
    """取得磁碟使用量（GB）"""
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f))
            except:
                pass
    return total / (1024**3)


print('✅ 工具函式載入完成')

## Step 4: 設定 URL 清單與執行參數

In [ ]:
# ── 4. 設定訓練參數 ──────────────────────────────────────────
# 📌 請在這裡填入你的 YouTube URL 清單和 API Key

import os

# OpenAI API Key (用於幽默技巧分析，可選)
os.environ['OPENAI_API_KEY'] = 'sk-...'  # ← 填入你的 Key

# ── URL 清單 ────────────────────────────────────────────────
# 方法 A：直接貼上 URL 清單
YOUTUBE_URLS = [
    # '中文脫口秀頻道範例',
    # 'https://www.youtube.com/watch?v=XXXXXXX',
]

# 方法 B：從 Drive 讀取 URL 清單檔
URL_LIST_FILE = f'{DRIVE_ROOT}/url_list.txt'
if os.path.exists(URL_LIST_FILE):
    with open(URL_LIST_FILE, 'r', encoding='utf-8') as f:
        file_urls = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    YOUTUBE_URLS = file_urls
    print(f'📋 從 Drive 讀到 {len(YOUTUBE_URLS)} 個 URL')
else:
    print(f'💡 提示：可以把 URL 清單上傳到 Drive: {URL_LIST_FILE}')
    print(f'   目前使用上方 YOUTUBE_URLS 清單 ({len(YOUTUBE_URLS)} 個)')

# ── 處理參數 ─────────────────────────────────────────────────
WHISPER_MODEL = 'large-v3'    # 或 'medium' 節省時間
LAUGHTER_THRESHOLD = 0.8      # YAMNet 信心閾值
FUNNY_THRESHOLD = 0.7         # Humor Score 閾值
ENABLE_VIDEO_ANALYSIS = False  # Colab T4 記憶體有限，建議先關閉
ENABLE_TECHNIQUE_ANALYSIS = (len(os.environ.get('OPENAI_API_KEY', '')) > 10)

# ── 路徑 ─────────────────────────────────────────────────────
PROCESSED_LOG = f'{DRIVE_ROOT}/processed_ids.json'  # 斷點續傳記錄
DATASET_PATH = f'{PROCESSED_DIR}/dataset.json'      # 最終資料集
CANDIDATES_PATH = f'{PROCESSED_DIR}/candidates.json' # 幽默候選集

print('\n📊 訓練設定:')
print(f'   Whisper 模型: {WHISPER_MODEL}')
print(f'   笑聲閾值: {LAUGHTER_THRESHOLD}')
print(f'   幽默閾值: {FUNNY_THRESHOLD}')
print(f'   視覺分析: {ENABLE_VIDEO_ANALYSIS}')
print(f'   技巧分析 (LLM): {ENABLE_TECHNIQUE_ANALYSIS}')

## Step 5: 資料處理管線（逐部影片，處理完即刪）

In [ ]:
# ── 5. 主管線：逐部影片處理 ─────────────────────────────────
# 核心設計：
#   - 每部影片下載 → 分析 → 結果存 Drive → 刪除音訊原檔
#   - 斷點續傳：已處理過的影片 ID 會被記錄，重跑時自動跳過

sys.path.insert(0, '/content/humor_src')  # 確保 src 路徑

# 延遲載入（避免影響其他 Cell 的執行）
print('🔄 載入處理模組...')
from humor_bot.data_engine.youtube_downloader import YouTubeDownloader
from humor_bot.data_engine.laughter_detector import LaughterDetector
from humor_bot.data_engine.audio_analyzer import AudioAnalyzer
from humor_bot.data_engine.alignment import SetupPunchlineAligner
from humor_bot.data_engine.auto_annotator import AutoAnnotationPipeline
print('✅ 模組載入完成')

# 初始化（只初始化一次，模型會常駐在 GPU 記憶體）
print('\n🤖 初始化 AI 模型（首次需要下載，約 2-5 分鐘）...')
downloader = YouTubeDownloader(
    output_dir=DATA_DIR + '/raw',
    whisper_model_size=WHISPER_MODEL,
)
detector = LaughterDetector(confidence_threshold=LAUGHTER_THRESHOLD)
analyzer = AudioAnalyzer()
aligner = SetupPunchlineAligner(min_laughter_confidence=LAUGHTER_THRESHOLD)
annotator = AutoAnnotationPipeline(
    funny_threshold=FUNNY_THRESHOLD,
    enable_video=ENABLE_VIDEO_ANALYSIS,
    enable_technique_analysis=ENABLE_TECHNIQUE_ANALYSIS,
)
print('✅ 所有 AI 模型初始化完成！')

# ── 載入現有資料集（繼續累積）────────────────────────────────
all_candidates = []
if os.path.exists(CANDIDATES_PATH):
    with open(CANDIDATES_PATH, 'r', encoding='utf-8') as f:
        all_candidates = json.load(f)
    print(f'📂 載入已有候選集 {len(all_candidates)} 筆')

already_processed = load_processed_ids(PROCESSED_LOG)
print(f'📋 已處理影片: {len(already_processed)} 部')

# ── 逐部影片處理 ──────────────────────────────────────────────
total = len(YOUTUBE_URLS)
success_count = 0
skip_count = 0
fail_count = 0

for i, url in enumerate(YOUTUBE_URLS, 1):
    # 提取 video ID 先判斷是否已處理
    import re
    vid_match = re.search(r'(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})', url)
    video_id_preview = vid_match.group(1) if vid_match else url[-15:]

    print(f'\n{'='*60}')
    print(f'[{i}/{total}] 🎬 {url}')

    if video_id_preview in already_processed:
        print(f'  ⏭️  已處理過，跳過')
        skip_count += 1
        continue

    # 磁碟空間檢查
    disk_gb = get_disk_usage_gb('/content')
    print(f'  💾 當前磁碟使用: {disk_gb:.1f} GB')
    if disk_gb > 60:  # Colab 約 80GB 上限，保留緩衝
        print('  ⚠️  磁碟空間不足，強制清理暫存...')
        for f in Path(f'{DATA_DIR}/raw/audio').glob('*.wav'):
            f.unlink()

    start_time = time.time()
    try:
        # ── Phase A: 下載音訊 + 字幕 ──────────────────────────
        print('  📥 Phase A: 下載音訊...')
        dl_result = downloader.process_url(url)
        video_id = dl_result.video_id
        print(f'  ✅ 下載完成: {dl_result.title[:40]} ({dl_result.duration:.0f}s)')
        print(f'     字幕來源: {dl_result.subtitle_source.value}')

        # ── Phase B: 笑聲偵測 ─────────────────────────────────
        print('  🎤 Phase B: 笑聲偵測 (YAMNet)...')
        laughter_events = detector.detect(dl_result.audio_path)
        print(f'     偵測到 {len(laughter_events)} 個笑聲事件')

        # ── Phase C: 音訊特徵分析 ─────────────────────────────
        audio_features = []
        for e in laughter_events:
            feat = analyzer.analyze_segment(
                dl_result.audio_path, e.start, e.end
            )
            audio_features.append({'start': feat.start, 'end': feat.end,
                                    'rms_db': feat.rms_db})

        # ── Phase D: Setup-Punchline 對齊 ────────────────────
        print('  📎 Phase D: Setup-Punchline 對齊...')
        if not (dl_result.transcript_path and
                dl_result.transcript_path.exists()):
            print('  ⚠️  無逐字稿，跳過此影片')
            cleanup_video_files(
                str(dl_result.audio_path),
                str(dl_result.subtitle_path) if dl_result.subtitle_path else None
            )
            fail_count += 1
            continue

        with open(dl_result.transcript_path, 'r', encoding='utf-8') as f:
            transcript_data = json.load(f)

        laughter_dicts = [
            {'start': e.start, 'end': e.end, 'duration': e.duration,
             'confidence': e.confidence, 'event_class': e.event_class}
            for e in laughter_events
        ]

        aligned_jokes = aligner.align(
            video_id, transcript_data, laughter_dicts, audio_features
        )
        print(f'     對齊到 {len(aligned_jokes)} 個段子')

        # ── Phase E: 自動標註 (Humor Score) ──────────────────
        print('  🏷️  Phase E: 自動標註...')
        candidates = annotator.run(
            video_id=video_id,
            aligned_jokes=aligned_jokes,
            laughter_events=laughter_dicts,
            audio_features=audio_features,
        )
        funny_count = sum(1 for c in candidates if c.auto_label == 'funny')
        print(f'     候選集: {len(candidates)} 筆 (😂 funny={funny_count})')

        # ── Phase F: 儲存結果 ─────────────────────────────────
        import dataclasses
        candidate_dicts = [dataclasses.asdict(c) for c in candidates]
        all_candidates.extend(candidate_dicts)

        # 存到本地暫存
        with open(CANDIDATES_PATH, 'w', encoding='utf-8') as f:
            json.dump(all_candidates, f, ensure_ascii=False, indent=2)

        # 同步到 Drive
        sync_to_drive(CANDIDATES_PATH, f'{DRIVE_ROOT}/candidates.json')

        # 單部影片結果也單獨存（方便審查）
        video_output = f'{PROCESSED_DIR}/{video_id}_candidates.json'
        with open(video_output, 'w', encoding='utf-8') as f:
            json.dump(candidate_dicts, f, ensure_ascii=False, indent=2)

        # ── Phase G: 刪除原始音訊（節省空間）────────────────
        print('  🧹 Phase G: 刪除原始音訊...')
        cleanup_video_files(
            str(dl_result.audio_path),
            str(dl_result.subtitle_path) if dl_result.subtitle_path else None
        )

        # 記錄已處理
        save_processed_id(video_id, PROCESSED_LOG)
        sync_to_drive(PROCESSED_LOG, f'{DRIVE_ROOT}/processed_ids.json')

        elapsed = time.time() - start_time
        success_count += 1
        print(f'  ✅ 完成！耗時 {elapsed:.0f}s | '
              f'累積候選集: {len(all_candidates)} 筆')

    except KeyboardInterrupt:
        print('\n⛔ 使用者中斷。已處理的資料已存到 Drive。')
        break
    except Exception as e:
        print(f'  ❌ 處理失敗: {e}')
        traceback.print_exc()
        fail_count += 1
        # 即使失敗也清理，避免佔用空間
        try:
            cleanup_video_files(
                f'{DATA_DIR}/raw/audio/{video_id_preview}.wav', None
            )
        except:
            pass

print(f'\n{"="*60}')
print(f'🎉 管線執行完畢！')
print(f'   ✅ 成功: {success_count} 部')
print(f'   ⏭️  跳過: {skip_count} 部（已處理）')
print(f'   ❌ 失敗: {fail_count} 部')
print(f'   📊 總候選集: {len(all_candidates)} 筆')
print(f'   💾 已存到 Drive: {DRIVE_ROOT}/')

## Step 6: 訓練 Reward Model

In [ ]:
# ── 6. 訓練 Reward Model ─────────────────────────────────────
# 使用 Step 5 產生的候選集訓練 Reward Model
# 模型：BERT-based 分類器，輸入 (setup, punchline)，預測 humor_score

print('🤖 訓練 Reward Model...')
print(f'   資料集: {len(all_candidates)} 筆候選')

import json
import numpy as np
from pathlib import Path

# 載入候選集
if not all_candidates:
    with open(f'{DRIVE_ROOT}/candidates.json', 'r', encoding='utf-8') as f:
        all_candidates = json.load(f)

# 準備訓練資料
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
import torch

# 過濾有效資料（有文本 + 有分數）
valid_data = [
    c for c in all_candidates
    if c.get('setup_text') and c.get('punchline_text')
    and c.get('humor_score') is not None
]
print(f'   有效訓練資料: {len(valid_data)} 筆')

if len(valid_data) < 50:
    print('⚠️  資料量不足（建議 > 500 筆），請先收集更多訓練資料')
else:
    # 建立 Dataset
    MODEL_NAME = 'bert-base-chinese'  # 針對中文
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def preprocess(examples):
        # 拼接 setup + [SEP] + punchline 作為輸入
        texts = [
            f"{s} [SEP] {p}"
            for s, p in zip(examples['setup_text'], examples['punchline_text'])
        ]
        tokenized = tokenizer(
            texts, padding='max_length', truncation=True, max_length=256
        )
        # 迴歸任務：humor_score 作為 label
        tokenized['labels'] = examples['humor_score']
        return tokenized

    dataset = Dataset.from_list(valid_data)
    dataset = dataset.train_test_split(test_size=0.15)
    tokenized_ds = dataset.map(preprocess, batched=True)

    # 模型（迴歸任務，num_labels=1）
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=1
    )

    # 訓練參數
    training_args = TrainingArguments(
        output_dir=f'{MODEL_DIR}/reward_model',
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_steps=100,
        weight_decay=0.01,
        logging_dir=f'{MODEL_DIR}/logs',
        logging_steps=50,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=2,
        report_to='none',  # 不用 wandb
    )

    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        preds = preds.squeeze()
        mse = np.mean((preds - labels) ** 2)
        mae = np.mean(np.abs(preds - labels))
        corr = np.corrcoef(preds, labels)[0, 1] if len(preds) > 1 else 0
        return {'mse': mse, 'mae': mae, 'pearson_r': corr}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds['train'],
        eval_dataset=tokenized_ds['test'],
        compute_metrics=compute_metrics,
    )

    print('\n🚀 開始訓練 Reward Model...')
    train_result = trainer.train()

    # 儲存模型到 Drive
    model_save_path = f'{MODEL_DIR}/reward_model_final'
    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)

    print(f'\n✅ Reward Model 訓練完成！')
    print(f'   模型儲存至: {model_save_path}')
    print(f'   訓練損失: {train_result.training_loss:.4f}')

## Step 7: 推論測試（輸入文稿預測幽默分數）

In [ ]:
# ── 7. 推論：輸入文稿預測幽默分數 ────────────────────────────
# 這就是專案的核心目標：輸入段子 → 預測觀眾會不會笑

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

MODEL_PATH = f'{MODEL_DIR}/reward_model_final'

print('📂 載入 Reward Model...')
try:
    rm_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    rm_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    rm_model.eval()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    rm_model.to(device)
    print(f'✅ 模型載入完成 (device: {device})')
except Exception as e:
    print(f'❌ 模型載入失敗: {e}')
    print('   請先執行 Step 6 訓練模型')
    rm_model = None

def predict_humor_score(setup: str, punchline: str) -> dict:
    """預測段子的幽默分數"""
    if rm_model is None:
        return {'error': '模型未載入'}

    text = f'{setup} [SEP] {punchline}'
    inputs = rm_tokenizer(
        text, return_tensors='pt',
        padding=True, truncation=True, max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = rm_model(**inputs)
        score = float(torch.sigmoid(outputs.logits[0][0]).cpu())

    # 解讀
    if score >= 0.7:
        label = '😂 很可能會笑'
    elif score >= 0.4:
        label = '🙂 可能會會心一笑'
    else:
        label = '😑 可能冷場'

    return {
        'humor_score': score,
        'prediction': label,
        'setup': setup,
        'punchline': punchline
    }


# ── 測試範例 ─────────────────────────────────────────────────
test_jokes = [
    {
        'setup': '我以前覺得婚姻是愛情的墳墓',
        'punchline': '現在發現不對，是我的自由的墳墓'
    },
    {
        'setup': '老師說要我們寫一篇關於環保的作文',
        'punchline': '我用正反兩面都寫了'
    },
]

print('\n🎯 幽默分數預測:')
print('='*60)
for joke in test_jokes:
    result = predict_humor_score(joke['setup'], joke['punchline'])
    print(f"\nSetup: {result['setup']}")
    print(f"Punchline: {result['punchline']}")
    print(f"→ Humor Score: {result.get('humor_score', 0):.3f} | {result.get('prediction', '')}")
    print('-'*40)

## Step 8: 從 YouTube 頻道批次爬取 URL

In [ ]:
# ── 8. 爬取 YouTube 頻道影片 URL ──────────────────────────────
# 用於自動從特定脫口秀頻道收集訓練影片

import yt_dlp
import re

# 🎯 填入你的目標頻道（可以是頻道主頁、播放清單、或搜尋關鍵字）
TARGET_CHANNELS = [
    # 台灣脫口秀頻道範例
    # 'https://www.youtube.com/@STRNetwork',
    # 'https://www.youtube.com/@gigclub',
    # 'https://www.youtube.com/playlist?list=XXXXXXX',
]

MAX_VIDEOS_PER_CHANNEL = 50  # 每個頻道最多抓幾部

def fetch_channel_urls(channel_url: str, max_videos: int) -> list:
    """從頻道/播放清單抓取影片 URL"""
    ydl_opts = {
        'extract_flat': True,
        'quiet': True,
        'playlistend': max_videos,
    }
    urls = []
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(channel_url, download=False)
        if 'entries' in info:
            for entry in info['entries']:
                if entry:
                    vid_id = entry.get('id', '')
                    if vid_id:
                        urls.append(f'https://www.youtube.com/watch?v={vid_id}')
    return urls

all_urls = []
for ch in TARGET_CHANNELS:
    print(f'📡 爬取: {ch}')
    try:
        urls = fetch_channel_urls(ch, MAX_VIDEOS_PER_CHANNEL)
        all_urls.extend(urls)
        print(f'   → {len(urls)} 支')
    except Exception as e:
        print(f'   ❌ 失敗: {e}')

# 去重
all_urls = list(set(all_urls))
print(f'\n✅ 共收集到 {len(all_urls)} 支不重複影片 URL')

# 儲存到 Drive
url_list_path = f'{DRIVE_ROOT}/url_list.txt'
with open(url_list_path, 'w', encoding='utf-8') as f:
    for u in all_urls:
        f.write(f'{u}\n')
print(f'💾 已儲存到 Drive: {url_list_path}')
print(f'   下一步：更新上方 YOUTUBE_URLS = [] 或直接重跑 Step 5')

## Step 9: 清理工作區（手動執行）

In [ ]:
# ── 9. 手動清理（謹慎使用）────────────────────────────────────
# 如果想強制清空所有暫存音訊（例如磁碟滿了），執行這 Cell

import shutil

CONFIRM_CLEANUP = False  # ← 改成 True 才會真的刪除

if CONFIRM_CLEANUP:
    audio_dir = Path(f'{DATA_DIR}/raw/audio')
    count = 0
    freed_mb = 0
    for f in audio_dir.iterdir():
        if f.is_file():
            size_mb = f.stat().st_size / (1024**2)
            f.unlink()
            freed_mb += size_mb
            count += 1
    print(f'🧹 已清理 {count} 個檔案，釋放 {freed_mb:.1f} MB')
else:
    audio_dir = Path(f'{DATA_DIR}/raw/audio')
    files = list(audio_dir.iterdir()) if audio_dir.exists() else []
    total_mb = sum(f.stat().st_size for f in files if f.is_file()) / (1024**2)
    print(f'📂 /content/data/raw/audio 目前有 {len(files)} 個檔案，佔 {total_mb:.1f} MB')
    print('   設 CONFIRM_CLEANUP = True 才會真的刪除')